# Opportunity Map Development

Based on the region profiling workflow, this notebook implements:
1. Geocoding an input area (e.g., "La Jolla, San Diego").
2. Finding the second-degree neighborhood block groups around the geocoded point.
3. Summarizing competitor business data (from the Google Maps scraper) including location and `review_rating`.
4. Calculating an Opportunity/Competition Score based on intersecting competitors to highlight areas with favorable conditions.
5. Visualizing the Opportunity Map.

In [2]:
%pip install shapely folium geopandas

  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached xyzservices-2025.11.0-py3-none-any.whl.metadata (4.3 kB)
Using cached folium-0.20.0-py2.py3-none-any.whl (113 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
Using cached xyzservices-2025.11.0-py3-none-any.whl (93 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import pandas as pd
import geopandas as gpd
import shapely
import folium
import warnings
from arcgis.gis import GIS
from arcgis.geocoding import geocode

warnings.filterwarnings('ignore')

# Connect to ArcGIS Online (Anonymous or with credentials if needed)
gis = GIS("https://ucsdonline.maps.arcgis.com/home", client_id="vWL6m9KaEcpoWNwQ")

Please sign in to your GIS and paste the code that is obtained below.
If a web browser does not automatically open, please navigate to the URL below yourself instead.
Opening web browser to navigate to: https://ucsdonline.maps.arcgis.com/sharing/rest/oauth2/authorize?response_type=code&client_id=vWL6m9KaEcpoWNwQ&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&state=85KgDDrBivwLV51DjRtKX8litm61Ur&allow_verification=false


## 1. Load Scraper Data
We extract the `latitude`, `longitude`, and `review_rating` directly from the GM-scraper dataset.

In [4]:
# Update path to your scraped communities data as needed
csv_path = "../gm-scraper/communities/lajolla.csv"

if os.path.exists(csv_path):
    df_businesses = pd.read_csv(csv_path)
else:
    # Dummy data fallback for demonstration
    print("Scraper dataset not found. Using dummy data.")
    df_businesses = pd.DataFrame({
        'title': ['Trilogy Sanctuary', 'Dummy Cafe', 'Test Restaurant'],
        'category': ['Vegan restaurant', 'Cafe', 'Restaurant'],
        'review_rating': [4.6, 4.2, 3.8],
        'latitude': [32.842986, 32.840000, 32.845000],
        'longitude': [-117.273729, -117.270000, -117.275000]
    })

# Convert to GeoDataFrame
gdf_businesses = gpd.GeoDataFrame(
    df_businesses, 
    geometry=gpd.points_from_xy(df_businesses.longitude, df_businesses.latitude),
    crs="EPSG:4326"
)
print(f"Loaded {len(gdf_businesses)} businesses.")

Loaded 76 businesses.


## 2. Block Groups and Neighborhood Definition
Defining the functions to geocode the input address and establish the study area.

In [5]:
BUFFER_SIZE = 500  # distance in feet (assuming CRS 2230 is in feet)

def get_pt(address):
    """Geocode an address and return a Shapely Point in EPSG:2230."""
    try:
        # Using ArcGIS geocoder
        results = geocode(address)
        if not results:
            return None
        p = results[0]['location']
    except Exception as e:
        print(f"Geocoding failed: {e}")
        return None
    
    # Convert EPSG:4326 (WGS84) to EPSG:2230 (NAD83 / California zone 6 ftUS)
    pt_series = gpd.GeoSeries([shapely.Point(p['x'], p['y'])], crs="EPSG:4326")
    return pt_series.to_crs("EPSG:2230").iloc[0]

def get_study_area(pt, bgs_shape):
    """Find the second-degree neighborhood block groups around a point."""
    # 1. Intersecting block group
    intersecting = bgs_shape[bgs_shape.geometry.intersects(pt)]
    if intersecting.empty:
        print("Point does not intersect any block group.")
        return gpd.GeoDataFrame(columns=bgs_shape.columns, crs=bgs_shape.crs)
    
    bg0 = intersecting.iloc[0].geometry
    
    # 2. First degree neighborhood (buffer + intersect)
    bg1_bgs = bgs_shape[bgs_shape.geometry.intersects(bg0.buffer(BUFFER_SIZE))]
    bg1 = bg1_bgs.dissolve().geometry.iloc[0]
    
    # 3. Second degree neighborhood (buffer + intersect)
    bg2 = bgs_shape[bgs_shape.geometry.intersects(bg1.buffer(BUFFER_SIZE))]
    return bg2.copy()

### Load Block Groups (Placeholder)
You should replace `bgs_shape` loading with your actual block group dataset (e.g., San Diego enriched block groups).

In [6]:
# Placeholder: Load your block groups dataset here
bg_path = "spatial_datasets/sd_block_groups.geojson"

if os.path.exists(bg_path):
    bgs_shape = gpd.read_file(bg_path).to_crs("EPSG:2230")
else:
    # Creating a synthetic grid of block groups around La Jolla for demonstration
    print("Block groups dataset not found. Generating a dummy 3x3 grid around La Jolla.")
    center_pt = get_pt("7650 Girard Ave, La Jolla, CA")
    grid_cells = []
    size = 1500
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            minx = center_pt.x + (dx * size) - (size/2)
            miny = center_pt.y + (dy * size) - (size/2)
            maxx = minx + size
            maxy = miny + size
            grid_cells.append(shapely.box(minx, miny, maxx, maxy))
            
    bgs_shape = gpd.GeoDataFrame(
        {'bg_id': range(len(grid_cells))}, 
        geometry=grid_cells, 
        crs="EPSG:2230"
    )

# Target Area
target_pt = get_pt("La Jolla, San Diego")
study_area_bgs = get_study_area(target_pt, bgs_shape)
print(f"Found {len(study_area_bgs)} block groups in the second-degree neighborhood.")

Block groups dataset not found. Generating a dummy 3x3 grid around La Jolla.
Found 9 block groups in the second-degree neighborhood.


## 3. Competition Score & Opportunity Map
For each block group, we intersect the competitors and average their Yelp/Google rating. 

*Opportunity Score Formula idea:*
High competition (many highly-rated competitors) = Low Opportunity.
Low competition (few competitors, or poorly-rated competitors) = High Opportunity.

In [7]:
def calculate_opportunity_score(study_area, businesses):
    # Ensure businesses are in the same CRS for spatial join
    biz_proj = businesses.to_crs(study_area.crs)
    
    # Spatial Join: Match businesses to the block group they fall into
    joined = gpd.sjoin(study_area, biz_proj, how="left", predicate="intersects")
    
    # Group by block group to calculate metrics
    stats = joined.groupby(joined.index).agg(
        competitor_count=('review_rating', 'count'),
        avg_rating=('review_rating', 'mean')
    )
    
    # Fill NaNs for block groups with 0 competitors
    stats['avg_rating'] = stats['avg_rating'].fillna(0)
    
    # Construct Opportunity Score
    # Simple metric: Maximum score is 100. 
    # We deduct points for the density of competitors and their quality.
    max_count = stats['competitor_count'].max() if stats['competitor_count'].max() > 0 else 1
    
    # Weighting: 50% based on competitor volume, 50% based on rating strength
    stats['opportunity_score'] = 100 - ( (stats['competitor_count'] / max_count) * 50 ) - ( (stats['avg_rating'] / 5.0) * 50 )
    
    # If 0 competitors, it gets a score of 100 (high opportunity)
    
    # Merge back
    study_area_scored = study_area.copy()
    study_area_scored['competitor_count'] = stats['competitor_count']
    study_area_scored['avg_rating'] = stats['avg_rating']
    study_area_scored['opportunity_score'] = stats['opportunity_score']
    
    return study_area_scored

scored_bgs = calculate_opportunity_score(study_area_bgs, gdf_businesses)
print(scored_bgs[['competitor_count', 'avg_rating', 'opportunity_score']].head())

   competitor_count  avg_rating  opportunity_score
0                 8    4.575000          37.583333
1                 0    0.000000         100.000000
2                 0    0.000000         100.000000
3                 7    4.614286          39.273810
4                15    4.473333          24.016667


## 4. Visualization

In [ ]:
map_bgs = scored_bgs.to_crs("EPSG:4326")

# center around la jolla
center_lat = map_bgs.geometry.centroid.y.mean()
center_lon = map_bgs.geometry.centroid.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

# opportunity choloropleth
folium.Choropleth(
    geo_data=map_bgs,
    name="Opportunity Score",
    data=map_bgs,
    columns=[map_bgs.index, "opportunity_score"],
    key_on="feature.id",
    fill_color="RdYlGn",
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name="Opportunity Score (Higher is Better)"
).add_to(m)

# plot the actual competitor businesses as markers
biz_in_area = gpd.sjoin(gdf_businesses, map_bgs, how="inner", predicate="within")
for idx, row in biz_in_area.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color="blue",
        fill=True,
        tooltip=f"{row['title']} - Rating: {row['review_rating']}"
    ).add_to(m)

folium.LayerControl().add_to(m)

# Display map
m